# Marker Gene Selection

In [ ]:
import pandas as pd
import numpy as np
from utils.misc import extract_number

In [ ]:
# load data
gene_expressions = pd.read_csv("data/train_data.csv", index_col=0)
gene_expressions_mat = gene_expressions.to_numpy()
genenames = np.array(gene_expressions.index.tolist())
samples = gene_expressions.columns.tolist()

# extract ages
ages = np.array([extract_number(timestring) for timestring in samples])
unique_ages=np.unique(ages)

# retain genes that are present in all samples
prevalence = np.mean(gene_expressions_mat > 0, axis=1)
subset_gene_id = np.where(prevalence == 1)[0]
subset_genenames = genenames[subset_gene_id]
gene_expressions = gene_expressions.loc[subset_genenames, :]
gene_expressions_mat = gene_expressions_mat[subset_gene_id, :]

# get log expressions
log_gene_expressions = np.log(gene_expressions)
log_gene_expressions_mat = np.log(gene_expressions_mat)

# transpose count tables to samples by genes
gene_expressions = gene_expressions.T
gene_expressions_mat = gene_expressions_mat.T
log_gene_expressions = log_gene_expressions.T
log_gene_expressions_mat = log_gene_expressions_mat.T

# get rankings of samples for each gene expression
gene_expressions_rank = log_gene_expressions.rank()

In [ ]:
nsamples = len(ages)

## Genes that are significantly different between samples of one age and all other ages

In [ ]:
from scipy.stats import wilcoxon, mannwhitneyu, spearmanr
from statsmodels.stats.multitest import multipletests
from utils.variation import var_comp
from scipy.stats import f, t

In [ ]:
pvals_singles = {}
unique_ages = np.unique(ages)

In [ ]:
for age in unique_ages:
    mask = np.array(ages == age)
    pvals_mat = np.zeros((2, gene_expressions.shape[1]))
    for j in range(gene_expressions.shape[1]):
        expression_singleage = gene_expressions_mat[mask, j]
        expression_others = gene_expressions_mat[np.logical_not(mask), j]
        _, pvals_mat[0, j] = mannwhitneyu(expression_singleage, expression_others, alternative='greater')
        _, pvals_mat[1, j] = mannwhitneyu(expression_singleage, expression_others, alternative='less')

    pvals_singles[age] = pvals_mat


In [ ]:
import pickle
with open("gene_plots/wilx_test_genes.pkl", 'wb+') as f:
    pickle.dump(pvals_singles, f)

In [ ]:
all_genes = gene_expressions.columns.to_numpy()

I call a gene marker gene if its wilcoxon rank sum test of gene expressions of one age against all others has a significant p value. The p value threshold is different depending on how many replicates there are for each age.

In [ ]:
pval_thresholds = {2:1e-3, 3: 5e-4, 4:1e-4, 5:5e-5}
marker_genes_singleage = []
for age in unique_ages:
    pval_threshold = pval_thresholds[np.sum(ages == age)]
    pvals_mat = pvals_singles[age]
    indices_more = np.where(pvals_mat[0, :] < pval_threshold )[0]
    indices_less = np.where(pvals_mat[1, :] < pval_threshold )[0]

    genes_more_df = pd.DataFrame({"Gene": all_genes[indices_more],
                                  f"Biggest_at_{age}": 1})
    genes_less_df = pd.DataFrame({"Gene": all_genes[indices_less],
                                  f"Smallest_at_{age}": 1})

    marker_genes_df = pd.merge(genes_more_df, genes_less_df, on="Gene", how="outer")
    marker_genes_df = marker_genes_df.fillna(0)

    marker_genes_singleage.append(marker_genes_df)

In [ ]:
from functools import reduce
def merge_dfs(left, right):
    return pd.merge(left, right, on="Gene", how='outer')

In [ ]:
marker_genes_singleage_combined = reduce(merge_dfs, marker_genes_singleage)
marker_genes_singleage_combined = marker_genes_singleage_combined.fillna(0)

In [ ]:
marker_genes_singleage_combined.to_csv("gene_plots/marker_genes_singleage.csv", index=False)

## Find genes that have strong variation between age 6 and 23
I first calculate F statistic for each gene to filter out genes that do not differ between samples of different ages between age 6 and 23. For the rest of the genes, I construct a correlation network and cluster the genes to identify genes of different time trends for prediction.

In [ ]:
ages_subset = ages[np.logical_and(ages >= 6, ages <= 23)]
gene_expressions_subset = gene_expressions.loc[np.logical_and(ages >= 6, ages <= 23), :]
gene_expressions_subset_rank = gene_expressions_subset.rank()

In [ ]:
variance_df = var_comp(gene_expressions_subset_rank, groups=ages_subset)
df1 = len(np.unique(ages_subset)) - 1
df2 = len(gene_expressions_subset_rank) - len(np.unique(ages_subset))

In [ ]:
from scipy.stats import f, t

In [ ]:
p_values_f = f.sf(variance_df["F_stat"], df1, df2)
reject_f, pvals_corrected_f, alphacSidak, alphacBonf = multipletests(p_values_f, alpha=0.05, method='fdr_bh')

In [ ]:
variance_df["P_Value"] = p_values_f

In [ ]:
subset_genes = variance_df.index[reject_f].tolist()
gene_expressions_subset_rank = gene_expressions_subset_rank.loc[:, subset_genes]
variance_df_subset = variance_df.loc[subset_genes, :]

For every gene, I calculate the mean expression at each age. For each gene, I normalize the expressions at the 10 timepoints to have a mean of 0 and sd of 1. Then I apply hierarchical clustering to the 1068 genes.

In [ ]:
gene_expressions_subset = gene_expressions_subset.loc[:, subset_genes]

In [ ]:
gene_expressions_subset["Age"] = ages_subset
log_gene_expressions_subset = np.log(gene_expressions_subset)
log_gene_expressions_byage = log_gene_expressions_subset.groupby('Age').mean()

In [ ]:
log_gene_expressions_byage_normalized = (log_gene_expressions_byage - log_gene_expressions_byage.mean()) / log_gene_expressions_byage.std()

In [ ]:
gene_profiles = log_gene_expressions_byage_normalized.to_numpy().T

In [ ]:
gene_profiles.shape

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
import matplotlib.pyplot as plt

In [ ]:
# calculate linkage
Z = linkage(gene_profiles, method='ward')

In [ ]:
plt.figure(figsize=(10, 5))
dendrogram(Z, no_labels=True)
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Genes")
plt.ylabel("Distance")
plt.savefig("gene_plots/markergene_6_23_hclust.pdf", format="pdf", bbox_inches="tight")

Based on the dendrogram above, I decide to split the genes into 10 clusters.

In [ ]:
clusters = fcluster(Z, t=10, criterion='maxclust')

In [ ]:
np.unique(clusters, return_counts=True)

In [ ]:
variance_df_subset["Cluster"] = clusters

In [ ]:
marker_genes_6_23 = []
for cnumber in np.arange(1, 11):
    selected_clusters = variance_df_subset.loc[variance_df_subset["Cluster"] == cnumber, :].sort_values(by="P_Value")
    cluster_genes = selected_clusters.index.tolist()
    marker_genes_6_23.extend(cluster_genes[0:int(len(cluster_genes)/5)])

In [ ]:
variance_df_subset.loc[marker_genes_6_23, :].to_csv("gene_plots/markergene_6_23.csv")

Plot all the marker genes

In [ ]:
from utils.viz import single_scatter_plot
from scipy.stats import rankdata
import matplotlib.pyplot as plt
import os

In [ ]:
gene_expressions.shape

In [ ]:
unique_ages = np.unique(ages)
age_rank = rankdata(ages, method='min').astype(int)
unique_ranks = np.unique(age_rank)
for genename in marker_genes_6_23:

    cid = variance_df_subset.loc[genename, 'Cluster']
    folder = f"gene_plots/predictor_genes/cluster_{cid}"
    values = gene_expressions.loc[:, genename].to_numpy()
    values = values.reshape(1, -1)
    fig = single_scatter_plot(ymat=values, xmat=age_rank.reshape(1, -1),
                            xticks=unique_ranks, xticknames=unique_ages.astype(str),
                            xname="Age (Month)", yname="Gene Expression (TPM)", title=genename)
    filename = f"{cid}_{genename}.pdf"
    fig.savefig(os.path.join(folder, filename), bbox_inches="tight")
    plt.close(fig)

# Decide on Panel of genes for prediction
Several considerations:
1. Gene expression at one age is distinct (bigger or smaller than) any other genes.
2. Gene expression varies greatly between age 6 month and 23 months.
3. Have genes with a variety of different time trends.


In [ ]:
marker_genes_singleage_combined = pd.read_csv("gene_plots/marker_genes_singleage.csv")
marker_genes_singleage_combined = marker_genes_singleage_combined.set_index('Gene')

In [ ]:
marker_genes_singleage_list = marker_genes_singleage_combined.index.tolist()
marker_genes_6_23_list = marker_genes_6_23_details.index.tolist()
marker_gene_intersection = np.intersect1d(marker_genes_singleage_list, marker_genes_6_23_list)

In [ ]:
counts_by_type_singleage = marker_genes_singleage_combined.sum()
counts_by_type_available = marker_genes_singleage_combined.loc[marker_gene_intersection, :].sum()

In [ ]:
counts_by_type_singleage

In [ ]:
marker_genes_singleage_combined.loc[marker_gene_intersection, :].sum()

Another angle for choosing potentially useful genes is to look for genes which has distinct gene expressions at a certain age (much bigger or smaller than gene expression at other ages). Among the current pool of genes selected based on variation between 6 and 23 months old, some of them have distinct expressions at age 2 and 4, but none of them have

Very few of the genes selected based on the variation between 6 and 23 months old have distinct gene expressions (bigger or smaller than gene expressions of other ages)

In [ ]:
marker_genes_candidates = variance_df_subset.loc[marker_gene_intersection, :]

In [ ]:
np.unique(marker_genes_candidates["Cluster"], return_counts=True)

In [ ]:
marker_genes_candidates.loc[marker_genes_candidates["Cluster"] == 5, :]